In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nikhil7280/weather-type-classification")

print("Path to dataset files:", path)

g:\DO_TOUCH\Programmes\.normal_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 186k/186k [00:00<00:00, 227kB/s]

Extracting files...
Path to dataset files: C:\Users\saran\.cache\kagglehub\datasets\nikhil7280\weather-type-classification\versions\1


In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
df = pd.read_csv("weather_classification_data.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13200 entries, 0 to 13199
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Temperature           13200 non-null  float64
 1   Humidity              13200 non-null  int64  
 2   Wind Speed            13200 non-null  float64
 3   Precipitation (%)     13200 non-null  float64
 4   Cloud Cover           13200 non-null  str    
 5   Atmospheric Pressure  13200 non-null  float64
 6   UV Index              13200 non-null  int64  
 7   Season                13200 non-null  str    
 8   Visibility (km)       13200 non-null  float64
 9   Location              13200 non-null  str    
 10  Weather Type          13200 non-null  str    
dtypes: float64(5), int64(2), str(4)
memory usage: 1.1 MB


In [ ]:
X = df.drop(columns=['Weather Type'])
y = df['Weather Type']

numeric_features = [
    'Temperature', 
    'Humidity', 
    'Wind Speed', 
    'Precipitation (%)', 
    'Atmospheric Pressure', 
    'UV Index', 
    'Visibility (km)'
]

categorical_features = [
    'Cloud Cover', 
    'Season', 
    'Location'
]

In [ ]:
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features),('cat', categorical_transformer, categorical_features)])

In [ ]:
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print("--- Random Forest Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))
print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred))

--- Random Forest Confusion Matrix ---
[[611  33   6  10]
 [ 37 601  14   8]
 [ 23  20 599  18]
 [ 28  17  13 602]]

--- Random Forest Classification Report ---
              precision    recall  f1-score   support

      Cloudy       0.87      0.93      0.90       660
       Rainy       0.90      0.91      0.90       660
       Snowy       0.95      0.91      0.93       660
       Sunny       0.94      0.91      0.93       660

    accuracy                           0.91      2640
   macro avg       0.92      0.91      0.91      2640
weighted avg       0.92      0.91      0.91      2640



In [8]:
encoded_cat_features = model_pipeline.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'] .get_feature_names_out(categorical_features)

all_features = numeric_features + list(encoded_cat_features)
importances = model_pipeline.named_steps['classifier'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n--- Feature Importance Ranking ---")
print(feature_importance_df.head(10).to_string(index=False))


--- Feature Importance Ranking ---
             Feature  Importance
         Temperature    0.202109
     Visibility (km)    0.143973
   Precipitation (%)    0.137696
            UV Index    0.131707
Atmospheric Pressure    0.115084
   Cloud Cover_clear    0.071905
            Humidity    0.054451
       Season_Winter    0.037430
          Wind Speed    0.034777
Cloud Cover_overcast    0.020295
